In [1]:
import os
import requests
import json
from pypdf import PdfReader
from langchain_community.llms import Ollama

from langchain_community.vectorstores import FAISS
from langchain.embeddings.base import Embeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.docstore.document import Document

/home/labuser/.local/lib/python3.8/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from this module in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [15]:
# Define the OllamaEmbeddings class
class OllamaEmbeddings(Embeddings):
    def __init__(self, model='llama3.2', url='http://localhost:11434/api/embeddings'):
        self.model = model
        self.url = url

    def embed(self, text):
        headers = {
            'Content-Type': 'application/json'
        }
        payload = {
            'model': self.model,
            'prompt': text
        }
        
        response = requests.post(self.url, headers=headers, data=json.dumps(payload))
        
        if response.status_code == 200:
            response_json = response.json()
            # Print the response structure for debugging
            print("Response JSON:", 1) #response_json)
            # Check the actual structure of the response and extract embeddings accordingly
            if 'embedding' in response_json:
                return response_json['embedding']
            else:
                print("Key 'embedding' not found in response")
                return None
        else:
            print(f"Error {response.status_code}: {response.text}")
            raise Exception(f"Error {response.status_code}: {response.text}")

    def embed_documents(self, texts):
        return [self.embed(text) for text in texts]

    def embed_query(self, text):
        return self.embed(text)


In [16]:
# Loading the LLM
llm = Ollama(
    model="llama3.2",
    temperature=0,
    # server="http://localhost:11434"  # Specify the server address
)

In [17]:
# Loading the document using PyPDF2
def load_pdf(file_path, num_pages=None):
    reader = PdfReader(file_path)
    text = ""
    pages = reader.pages[:num_pages] if num_pages else reader.pages
    for page in pages:
        text += page.extract_text() + "\n"
    return text

pdf_path = "tcsbancs-overarching-brochure-2023.pdf"
document_text = load_pdf(pdf_path, num_pages=5)

In [18]:
# Split the document into chunks
text_splitter = CharacterTextSplitter(separator="\n",
                                      chunk_size=256,
                                      chunk_overlap=32)

text_chunks = text_splitter.split_text(document_text)

In [19]:
# Convert text chunks to document objects
documents = [Document(page_content=chunk) for chunk in text_chunks]




In [20]:
# Loading the vector embedding model
embeddings = OllamaEmbeddings()



In [21]:
# Create knowledge base
knowledge_base = FAISS.from_documents(documents, embeddings)



Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1
Response JSON: 1


In [22]:
# Retrieval QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=knowledge_base.as_retriever()
)



In [23]:
question = "What is this document about?"
response = qa_chain.invoke({"query": question})
print(response["result"])

Response JSON: 1
This document appears to be about a financial services company, likely a bank or financial institution, and its various services, technologies, and innovations in the financial sector. It mentions specific areas such as wealth and asset management, private banking, and Banking-as-a-Service, suggesting that it may be a marketing or sales document aimed at highlighting the company's capabilities and offerings.


### Sample Prompt for Summarization and Extraction 


In [24]:
question_prompt = f"""
    Based on the document provided, perform the following tasks.

    Task:
    1. Summarize the document by providing a concise summary of the main points, focusing on the features, functionalities, \
    and benefits of the TCS BaNCS solution.
    2. Extract all the important information that describe how the solution has helped various banks like SBI and influenced industries like Capital \
    Markets in foreign countries. Extract as much information as possible.

    Provide the summary first, followed by the extracted information.

    Summary:
    """
response = qa_chain.invoke({"query": question_prompt})
print(response["result"])

Response JSON: 1
I don't know the answer to your question based on the provided context. The document appears to be describing a financial services solution called TCS BaNCS, but it does not mention SBI or its influence on Capital Markets in foreign countries. There is no information about how the solution has helped specific banks like SBI or influenced industries like Capital Markets in foreign countries.


### Converting Summary and Extracted Metrics into HTML format (Sample Task Completion Scenario)

In [25]:
question_prompt = f"""
    Based on the document provided, perform the following tasks.

    Task:
    1. Summarize the document by providing a concise summary of the main points, focusing on the features,\
    functionalities, and benefits of the TCS BaNCS solution.
    2. Extract all the important information that describe how the solution has helped various banks like SBI \
    and influenced industries like Capital Markets in foreign countries. Extract as much information as possible. \
    Provide the summary first, followed by the extracted information.
    3. Format everything as HTML content which can be used to build a website. Provide the title at the top \
    of the webpage as "TCS BaNCS Overview". Place the summary in <div> tags.
    4. Create a table titled Key Metrics and place the information extracted as rows of data. Use bold tags \
    to highlight crucial elements like names and numbers. 
    
    HTML Code:
    """
response = qa_chain.invoke({"query": question_prompt})
print(response["result"])

Response JSON: 1
I don't know the answer to your question based on the provided context. The document doesn't mention SBI or its influence on industries like Capital Markets in foreign countries. It only provides information about the features, functionalities, and benefits of the TCS BaNCS solution.

However, I can provide a summary of the document and extract important information that describes the features and benefits of the TCS BaNCS solution:

**Summary:**
The TCS BaNCS solution is a comprehensive suite of services that provides wealth and asset management, private banking, and payment operations. It offers quicker processing times, low latency algorithmic trading capabilities, and connectivity to SWIFT, SWIFTgpi, and local or regional payments infrastructure. The solution is tailored to facilitate informed decision making, meet dynamic customer demands, and regulatory expectations.

**Extracted Information:**

* Transformations in Asia
* Processing more than 1 billion accounts


### Load Python libraries to view HTML

In [26]:
from IPython.display import display, HTML
display(HTML(response['result']))

Feature/Functionality,Description
Transformations in Asia,Processing more than 1 billion accounts
Preferred by credit unions and community banks in the UK and USA,Facilitating Banking-as-a-Service for many customers
Comprehensive suite of services,Encompassing wealth and asset management and private banking
Quicker processing times,Low latency algorithmic trading capabilities
"Connectivity to SWIFT, SWIFTgpi, and local or regional payments infrastructure","Tailored to facilitate informed decision making, meet dynamic customer demands, and regulatory expectations"
